# 03 — RAG Chain & Model Serving Deployment

**Purpose:** Wire the Vector Search retriever to a Foundation Model API chat model to answer
questions about the insurance data, then deploy that chain as a real Mosaic AI Model Serving
endpoint — this is the "RAG served with Mosaic AI Model Serving" deliverable.

### What This Notebook Does
1. Defines an `mlflow.pyfunc.ChatAgent` that retrieves top-k policy documents and grounds a
   Foundation Model API chat completion in them.
2. Smoke-tests the agent locally, then logs and registers it to Unity Catalog with MLflow.
3. Deploys the registered model via the Agent Framework to a Mosaic AI Model Serving
   endpoint and queries it live to prove it works end-to-end.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | `insurance_rag_agent.docs.policy_documents_index` |
| Target | UC model `insurance_rag_agent.agent_tools.insurance_rag_model` + serving endpoint |

> **Prerequisites:** Run `src/02_vector_search_index` first. Deploying consumes Free
> Edition's limited model-serving quota — stop/delete the endpoint after demoing.

---

In [ ]:
%pip install -U mlflow databricks-agents databricks-vectorsearch openai
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG               = 'insurance_rag_agent'
VS_ENDPOINT_NAME       = 'insurance_rag_agent_vs_endpoint'
INDEX_NAME             = f'{CATALOG}.docs.policy_documents_index'
# Get CHAT_MODEL_ENDPOINT and MODEL_QUERY_BASE_PATH from Serving > (your chat model) >
# "Use this model" > the model= and base_url= values in the generated code snippet.
# AI-Gateway/UC-governed workspaces show something like:
#   model='insurance_rag_agent.agent_tools.meta-llama-3-3-70b-instruct_backend'
#   base_url='https://<host>/ai-gateway/mlflow/v1'
# Classic Foundation Model API workspaces show something like:
#   model='databricks-meta-llama-3-3-70b-instruct'
#   base_url='https://<host>/serving-endpoints'
CHAT_MODEL_ENDPOINT    = f'{CATALOG}.agent_tools.meta-llama-3-3-70b-instruct_backend'
MODEL_QUERY_BASE_PATH  = '/ai-gateway/mlflow/v1'
RAG_MODEL_NAME         = f'{CATALOG}.agent_tools.insurance_rag_model'
RAG_ENDPOINT_NAME      = 'insurance_rag_endpoint'
NUM_RESULTS            = 3
SAMPLE_QUESTION        = 'What do the records show about smokers in the southeast region?'

print(f'Index: {INDEX_NAME}')
print(f'Chat model endpoint: {CHAT_MODEL_ENDPOINT}')
print(f'Registered model: {RAG_MODEL_NAME}')

In [0]:
import uuid

import mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse

mlflow.set_registry_uri('databricks-uc')

---

In [ ]:
class InsuranceRAGAgent(ChatAgent):
    '''Retrieves relevant policy documents, then grounds a Foundation Model API chat
    completion in them. Clients are created inside predict() rather than __init__ so
    the agent instance stays picklable for MLflow logging.'''

    def __init__(self, endpoint_name, index_name, chat_model_endpoint, base_path, num_results):
        self.endpoint_name       = endpoint_name
        self.index_name          = index_name
        self.chat_model_endpoint = chat_model_endpoint
        self.base_path           = base_path
        self.num_results         = num_results

    def predict(self, messages, context=None, custom_inputs=None):
        from databricks.sdk import WorkspaceClient
        from databricks.vector_search.client import VectorSearchClient
        from openai import OpenAI

        question = messages[-1].content

        vsc   = VectorSearchClient()
        index = vsc.get_index(endpoint_name=self.endpoint_name, index_name=self.index_name)
        results = index.similarity_search(
            query_text  = question,
            columns     = ['policy_text'],
            num_results = self.num_results,
        )
        context_text = '\n'.join(row[0] for row in results['result']['data_array'])

        # This workspace proxies Foundation Model calls through AI Gateway rather than the
        # classic serving_endpoints.query() path, so we use the OpenAI-compatible client.
        # config.authenticate() (not config.token, which is only set for PAT auth) returns a
        # fresh Authorization header regardless of auth method — notebook OAuth, PAT, or the
        # credentials injected into a deployed Model Serving container.
        w           = WorkspaceClient()
        bearer_token = w.config.authenticate()['Authorization'].split(' ', 1)[1]
        openai_client = OpenAI(
            api_key  = bearer_token,
            base_url = f'{w.config.host}{self.base_path}',
        )
        completion = openai_client.chat.completions.create(
            model    = self.chat_model_endpoint,
            messages = [
                {'role': 'system', 'content': f'Answer using only this context:\n{context_text}'},
                {'role': 'user',   'content': question},
            ],
        )
        answer = completion.choices[0].message.content

        return ChatAgentResponse(messages=[
            ChatAgentMessage(id=str(uuid.uuid4()), role='assistant', content=answer)
        ])


rag_agent = InsuranceRAGAgent(
    endpoint_name       = VS_ENDPOINT_NAME,
    index_name          = INDEX_NAME,
    chat_model_endpoint = CHAT_MODEL_ENDPOINT,
    base_path           = MODEL_QUERY_BASE_PATH,
    num_results         = NUM_RESULTS,
)

smoke_test = rag_agent.predict(
    [ChatAgentMessage(id=str(uuid.uuid4()), role='user', content=SAMPLE_QUESTION)]
)
print(smoke_test.messages[-1].content)

---

In [0]:
with mlflow.start_run(run_name='insurance_rag_agent'):
    logged_model = mlflow.pyfunc.log_model(
        name                  = 'rag_chain',
        python_model          = rag_agent,
        registered_model_name = RAG_MODEL_NAME,
    )

model_version = logged_model.registered_model_version
print(f'Registered {RAG_MODEL_NAME} version {model_version}')

---

In [ ]:
from databricks import agents

deployment_info = agents.deploy(
    model_name    = RAG_MODEL_NAME,
    model_version = model_version,
    endpoint_name = RAG_ENDPOINT_NAME,
)

print(f'Deployed endpoint: {deployment_info.endpoint_name}')

---

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

live_response = w.serving_endpoints.query(
    name     = deployment_info.endpoint_name,
    messages = [{'role': 'user', 'content': SAMPLE_QUESTION}],
)

print(live_response.choices[0].message.content)